# BRFSS 2015 - data preparation pipeline (de-duplicated variant)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/03_brfss_data_pipeline.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Removes the 24,206 repeated rows, splits 70/15/15, scales on training data, applies a custom SMOTE-NC to training only and ranks features. Unfinished modelling cells from the original notebook were removed.

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


# Early Diabetes Prediction — Data Preparation Pipeline

**Dataset:** Diabetes Health Indicators (BRFSS 2015), Kaggle (Alex Teboul)

This notebook reproduces every step described in the accompanying research report:

1. Data collection & framing (load, type, document schema)
2. Preprocessing (validity checks, dedup, casting)
3. Class distribution check
4. Distributional analysis (EDA) — figures saved to `./output/figures`
5. Stratified 70/15/15 split
6. Normalization (`StandardScaler` fit on **train only**)
7. SMOTE balancing (**training set only** — custom SMOTE-NC style implementation; `imbalanced-learn` is not required)
8. Feature selection analysis (Correlation, RF importance, Information Gain, Gain Ratio, Mutual Information)

> **Converted from `diabetes_pipeline.py` for Google Colab.** The original used `argparse` for command-line arguments and a non-interactive `Agg` matplotlib backend — both of which don't work in a notebook. Below, arguments become a plain configuration cell, and plots render inline.

**How to run:** Run cells top to bottom. Use the upload cell to bring in your CSV, or edit `INPUT_CSV` to point at a file already in your Colab environment / mounted Drive.


In [ ]:
# If running on a fresh Colab runtime, these are already installed.
# Uncomment the line below only if you get a "No module named ..." error.
# !pip install -q scikit-learn scipy seaborn matplotlib pandas numpy

import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sstats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

%matplotlib inline


### Upload the dataset (Colab only)

Run the cell below to upload `diabetes_binary_health_indicators_BRFSS2015.csv` from your computer. If you're not on Colab, or already have the file available, skip this cell and just set `INPUT_CSV` in the next cell.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        INPUT_CSV = list(uploaded.keys())[0]
        print(f"Uploaded: {INPUT_CSV}")
except ImportError:
    print("Not running in Colab — set INPUT_CSV manually in the next cell.")


In [ ]:
# ---- Configuration (replaces argparse for notebook use) ----
# If you used the upload cell above, INPUT_CSV is already set. Otherwise, set it here:
try:
    INPUT_CSV
except NameError:
    INPUT_CSV = "diabetes_binary_health_indicators_BRFSS2015.csv"

OUTDIR = "./output"
FIGDIR = os.path.join(OUTDIR, "figures")
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

RANDOM_STATE = 42
TARGET = "Diabetes_binary"

BINARY_COLS = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
               'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
               'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
ORDINAL_COLS = ['GenHlth', 'Age', 'Education', 'Income']
CONT_COLS = ['BMI', 'MentHlth', 'PhysHlth']          # true-continuous / count columns
CONT_ORD_COLS = CONT_COLS[:] + ORDINAL_COLS          # columns that get standardized
CATEG_COLS_FOR_SMOTE = BINARY_COLS + ORDINAL_COLS    # columns SMOTE majority-votes on

K_NEIGHBORS = 5

print("INPUT_CSV:", INPUT_CSV)
print("OUTDIR:", OUTDIR)


## Step 1–2: Data Collection, Framing & Preprocessing

In [ ]:
def clean_data(raw_path: str):
    df = pd.read_csv(raw_path)
    report = {
        "raw_shape": list(df.shape),
        "raw_missing_cells": int(df.isnull().sum().sum()),
        "raw_duplicate_rows": int(df.duplicated().sum()),
    }

    # --- validity / range checks against the BRFSS 2015 codebook ---
    violations = {}
    for c in BINARY_COLS + [TARGET]:
        bad = (~df[c].isin([0.0, 1.0])).sum()
        if bad:
            violations[c] = int(bad)
    bounds = {"GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8),
              "MentHlth": (0, 30), "PhysHlth": (0, 30)}
    for c, (lo, hi) in bounds.items():
        bad = (~df[c].between(lo, hi)).sum()
        if bad:
            violations[c] = int(bad)
    bad = (df["BMI"] <= 0).sum()
    if bad:
        violations["BMI_nonpositive"] = int(bad)
    report["range_violations"] = violations

    # --- drop exact duplicate rows (kept as first occurrence) ---
    before = len(df)
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    report["duplicates_removed"] = int(before - len(df))
    report["missing_after_clean"] = int(df.isnull().sum().sum())

    # --- cast to correct dtypes ---
    for c in BINARY_COLS + [TARGET]:
        df[c] = df[c].astype("int8")
    for c in ORDINAL_COLS + ["MentHlth", "PhysHlth"]:
        df[c] = df[c].astype("int16")
    df["BMI"] = df["BMI"].astype("float32")

    report["bmi_gt_60_count"] = int((df["BMI"] > 60).sum())
    report["clean_shape"] = list(df.shape)
    report["class_counts_after_clean"] = df[TARGET].value_counts().to_dict()
    return df, report


In [ ]:
print(">>> Step 1-2: Cleaning & framing")
df, clean_report = clean_data(INPUT_CSV)
df.to_csv(os.path.join(OUTDIR, "clean_full_dataset.csv"), index=False)
print(json.dumps(clean_report, indent=2, default=str))
df.head()


## Step 3–4: Class Distribution & Distributional Analysis (EDA)

In [ ]:
def run_eda(df: pd.DataFrame, figdir: str) -> dict:
    os.makedirs(figdir, exist_ok=True)
    for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
        if style in plt.style.available:
            plt.style.use(style)
            break
    plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.titlesize": 14,
                          "axes.titleweight": "bold"})
    BLUE, ORANGE = "#4C72B0", "#DD8452"
    stats = {}

    counts = df[TARGET].value_counts().sort_index()
    pct = df[TARGET].value_counts(normalize=True).sort_index() * 100
    stats["class_counts"] = counts.to_dict()
    stats["class_pct"] = pct.round(2).to_dict()
    stats["imbalance_ratio"] = round(counts[0] / counts[1], 2)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    bars = ax.bar(["No Diabetes (0)", "Diabetes (1)"], counts.values, color=[BLUE, ORANGE])
    for b, c, p_ in zip(bars, counts.values, pct.values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2500, f"{c:,}\n({p_:.1f}%)",
                ha="center", fontweight="bold")
    ax.set_title(f"Class Distribution (imbalance ~ {stats['imbalance_ratio']}:1)")
    ax.set_ylabel("Respondents")
    plt.tight_layout()
    plt.savefig(f"{figdir}/01_class_distribution.png", bbox_inches="tight")
    plt.show()
    plt.close()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, col in zip(axes, CONT_COLS):
        ax.hist(df[col], bins=40, color=BLUE, edgecolor="white", alpha=0.85)
        ax.axvline(df[col].mean(), color="red", ls="--", label=f"Mean {df[col].mean():.1f}")
        ax.axvline(df[col].median(), color="green", ls="--", label=f"Median {df[col].median():.1f}")
        ax.set_title(col); ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{figdir}/02_continuous_distributions.png", bbox_inches="tight")
    plt.show()
    plt.close()
    stats["skewness"] = {c: round(float(df[c].skew()), 3) for c in CONT_COLS}

    corr = df.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(13, 11))
    sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f",
                annot_kws={"size": 7}, ax=ax)
    plt.tight_layout()
    plt.savefig(f"{figdir}/05_correlation_heatmap.png", bbox_inches="tight")
    plt.show()
    plt.close()

    return stats


In [ ]:
print("\n>>> Step 3-4: EDA")
eda_report = run_eda(df, FIGDIR)
print(json.dumps(eda_report, indent=2, default=str))


## Step 5–6: Stratified 70/15/15 Split & Normalization

`StandardScaler` is fit on the **training split only**, then applied to val/test — this avoids leaking information from val/test into the scaling.

In [ ]:
def split_and_scale(df: pd.DataFrame):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

    scaler = StandardScaler()
    X_train_s, X_val_s, X_test_s = X_train.copy(), X_val.copy(), X_test.copy()
    X_train_s[CONT_ORD_COLS] = scaler.fit_transform(X_train[CONT_ORD_COLS])
    X_val_s[CONT_ORD_COLS] = scaler.transform(X_val[CONT_ORD_COLS])
    X_test_s[CONT_ORD_COLS] = scaler.transform(X_test[CONT_ORD_COLS])

    train_df = X_train_s.assign(**{TARGET: y_train.values})
    val_df = X_val_s.assign(**{TARGET: y_val.values})
    test_df = X_test_s.assign(**{TARGET: y_test.values})
    return train_df, val_df, test_df, scaler


In [ ]:
print("\n>>> Step 5-6: Split + normalize")
train_df, val_df, test_df, scaler = split_and_scale(df)
train_df.to_csv(os.path.join(OUTDIR, "train_scaled_presmote.csv"), index=False)
val_df.to_csv(os.path.join(OUTDIR, "val_scaled.csv"), index=False)
test_df.to_csv(os.path.join(OUTDIR, "test_scaled.csv"), index=False)
print(f"train={train_df.shape} val={val_df.shape} test={test_df.shape}")


## Step 7: SMOTE Balancing (Training Set Only)

Custom SMOTE-NC style implementation — continuous features are linearly interpolated between a minority sample and one of its nearest minority neighbors; categorical/ordinal features are set by majority vote among the same neighbors. No `imbalanced-learn` dependency required.

In [ ]:
def smote_balance(train_df: pd.DataFrame, k=K_NEIGHBORS, random_state=RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    feature_cols = [c for c in train_df.columns if c != TARGET]
    X = train_df[feature_cols].values.astype(float)
    y = train_df[TARGET].values.astype(int)
    col_idx = {c: i for i, c in enumerate(feature_cols)}
    cont_idx = [col_idx[c] for c in CONT_COLS]
    categ_idx = [col_idx[c] for c in CATEG_COLS_FOR_SMOTE]

    minority_label = int(pd.Series(y).value_counts().idxmin())
    X_min = X[y == minority_label]
    n_min = (y == minority_label).sum()
    n_maj = (y != minority_label).sum()
    n_gen = int(n_maj - n_min)
    if n_gen <= 0:
        return train_df.copy()

    dist_scaler = StandardScaler()
    X_min_dist = dist_scaler.fit_transform(X_min)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min_dist)
    _, neigh_idx_all = nn.kneighbors(X_min_dist)
    neigh_idx_all = neigh_idx_all[:, 1:]

    base_choices = rng.integers(0, n_min, size=n_gen)
    nn_slot = rng.integers(0, k, size=n_gen)
    neighbor_choices = neigh_idx_all[base_choices, nn_slot]
    gaps = rng.random(n_gen)

    synthetic = np.empty((n_gen, X.shape[1]))
    base_cont = X_min[base_choices][:, cont_idx]
    neigh_cont = X_min[neighbor_choices][:, cont_idx]
    synthetic[:, cont_idx] = base_cont + gaps[:, None] * (neigh_cont - base_cont)

    for ci in categ_idx:
        neighbor_block = X_min[neigh_idx_all[base_choices]][:, :, ci]
        base_col = X_min[base_choices, ci].reshape(-1, 1)
        votes = np.concatenate([base_col, neighbor_block], axis=1)
        synthetic[:, ci] = sstats.mode(votes, axis=1, keepdims=False).mode

    synth_df = pd.DataFrame(synthetic, columns=feature_cols)
    synth_df[TARGET] = minority_label
    balanced = pd.concat([train_df, synth_df], ignore_index=True)
    return balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


In [ ]:
print("\n>>> Step 7: SMOTE (training set only)")
train_smote = smote_balance(train_df)
train_smote.to_csv(os.path.join(OUTDIR, "train_smote_balanced.csv"), index=False)
print(train_smote[TARGET].value_counts())


## Step 8: Feature Selection Analysis

Ranks features by Pearson correlation, Random Forest importance, Information Gain, Gain Ratio, and Mutual Information, then averages the ranks.

In [ ]:
def feature_selection(train_df: pd.DataFrame, random_state=RANDOM_STATE) -> pd.DataFrame:
    X = train_df.drop(columns=[TARGET])
    y = train_df[TARGET]
    feature_cols = list(X.columns)
    res = pd.DataFrame(index=feature_cols)

    corrs = X.assign(**{TARGET: y}).corr(numeric_only=True)[TARGET].drop(TARGET)
    res["Pearson_r"] = corrs
    res["Abs_Pearson_r"] = corrs.abs()

    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                 class_weight="balanced", random_state=random_state, n_jobs=-1)
    rf.fit(X, y)
    res["RF_Importance"] = rf.feature_importances_

    def entropy(labels):
        _, counts = np.unique(labels, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log2(p))

    def info_gain(vals, labels):
        h = entropy(labels)
        for v in np.unique(vals):
            subset = labels[vals == v]
            h -= (len(subset) / len(labels)) * entropy(subset)
        return h

    def split_info(vals):
        _, counts = np.unique(vals, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log2(p))

    y_arr = y.values
    ig_list, gr_list = [], []
    for c in feature_cols:
        vals = pd.qcut(X[c], q=10, duplicates="drop").cat.codes.values if c in CONT_COLS else X[c].values
        ig = info_gain(vals, y_arr)
        si = split_info(vals)
        ig_list.append(ig)
        gr_list.append(ig / si if si > 0 else 0.0)
    res["Information_Gain"] = ig_list
    res["Gain_Ratio"] = gr_list

    discrete_mask = [c not in CONT_COLS for c in feature_cols]
    res["Mutual_Information"] = mutual_info_classif(X, y, discrete_features=discrete_mask,
                                                      random_state=random_state)

    rank_cols = ["Abs_Pearson_r", "RF_Importance", "Information_Gain", "Mutual_Information"]
    for rc in rank_cols:
        res[rc + "_rank"] = res[rc].rank(ascending=False)
    res["Avg_Rank"] = res[[rc + "_rank" for rc in rank_cols]].mean(axis=1)
    return res.sort_values("Avg_Rank")


In [ ]:
print("\n>>> Step 8: Feature selection")
fs_results = feature_selection(train_df)
fs_results.to_csv(os.path.join(OUTDIR, "feature_selection_results.csv"))
fs_results.round(4)


## Done

All artifacts (cleaned data, splits, SMOTE-balanced training set, feature-selection results, and EDA figures) have been written under `OUTDIR`.

In [ ]:
print(f"All artifacts written to: {OUTDIR}")
for root, _, files_ in os.walk(OUTDIR):
    for fn in files_:
        print(os.path.join(root, fn))
